# Airbnb NYC — Data Cleaning & EDA
### Python Cleaning Pipeline | Before Loading to PostgreSQL

**Dataset:** Airbnb Open Data — New York City  
**Rows:** 102,599 | **Columns:** 26  
**Goal:** Understand, clean and prepare data for PostgreSQL + Booking Analysis

---


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully.")

---
## 2. Load Raw Dataset

We load everything as `string (dtype='str')` first.  
This is important — if we let pandas guess types, it will misread  
the `price` column (which has `$` signs) as objects, and may  
silently coerce things we haven't inspected yet.


In [ ]:
df = pd.read_csv('Airbnb_Open_Data.csv', dtype='str')

print(f"Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print()
print("Columns:", df.columns.tolist())

In [ ]:
# First look at the data
df.head(3)

---
## 3. Basic Dataset Info


In [ ]:
df.info()

In [ ]:
# Shape summary
print(f"Total Rows    : {df.shape[0]:,}")
print(f"Total Columns : {df.shape[1]}")

---
## 4. Null Value Analysis

Understanding which columns have missing data — and how much —  
helps us decide whether to **drop**, **fill**, or **remove the column entirely**.


In [ ]:
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)

null_report = pd.DataFrame({
    'Null Count' : null_counts,
    'Null %'     : null_pct
}).sort_values('Null %', ascending=False)

null_report[null_report['Null Count'] > 0]

In [ ]:
# Visualise null percentages
fig, ax = plt.subplots(figsize=(10, 6))

cols_with_nulls = null_report[null_report['Null Count'] > 0]
colors = ['#E24B4A' if p > 40 else '#EF9F27' if p > 5 else '#1D9E75'
          for p in cols_with_nulls['Null %']]

bars = ax.barh(cols_with_nulls.index, cols_with_nulls['Null %'], color=colors)

ax.set_xlabel('Null Percentage (%)', fontsize=12)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
ax.axvline(x=40, color='#E24B4A', linestyle='--', alpha=0.5, label='40% threshold (drop column)')
ax.axvline(x=5,  color='#EF9F27', linestyle='--', alpha=0.5, label='5% threshold (fill)')
ax.legend(fontsize=10)

for bar, pct in zip(bars, cols_with_nulls['Null %']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("Red   = Drop the column (too many nulls, no value)")
print("Amber = Fill with appropriate value")
print("Green = Small null count, easy to handle")

---
## 5. Duplicate Row Check


In [ ]:
total_dupes  = df.duplicated().sum()
id_dupes     = df['id'].duplicated().sum()

print(f"Fully duplicate rows : {total_dupes}")
print(f"Duplicate listing IDs: {id_dupes}")
print()

# Preview a duplicate example
if id_dupes > 0:
    dup_id = df[df['id'].duplicated(keep=False)]['id'].iloc[0]
    print(f"Example — listing ID '{dup_id}' appears multiple times:")
    display(df[df['id'] == dup_id][['id', 'NAME', 'neighbourhood group', 'price']].head(3))

---
## 6. Categorical Column Inspection

Checking unique values in key categorical columns to catch  
**typos**, **inconsistent casing**, and **unexpected categories**.


In [ ]:
cat_cols = [
    'neighbourhood group', 'room type',
    'cancellation_policy', 'instant_bookable',
    'host_identity_verified', 'country'
]

for col in cat_cols:
    print(f"── {col} ──")
    print(df[col].value_counts(dropna=False).to_string())
    print()

In [ ]:
# Visualise room type and neighbourhood group distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Room type
room_counts = df['room type'].value_counts()
axes[0].bar(room_counts.index, room_counts.values,
            color=['#1D9E75','#378ADD','#EF9F27','#E24B4A'])
axes[0].set_title('Room Type Distribution (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Room Type')
axes[0].set_ylabel('Count')
for i, v in enumerate(room_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=10)

# Borough (after we see the typos)
borough_counts = df['neighbourhood group'].value_counts(dropna=False)
colors_b = ['#E24B4A' if str(b) in ['brookln','manhatan','nan'] else '#378ADD'
            for b in borough_counts.index]
axes[1].barh(borough_counts.index.astype(str), borough_counts.values, color=colors_b)
axes[1].set_title('Borough Distribution — Typos in Red', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')
for i, v in enumerate(borough_counts.values):
    axes[1].text(v + 100, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Numeric Column Inspection (Pre-Cleaning)

We inspect key numeric columns for outliers and impossible values  
**before** converting types — so we can see the raw problems clearly.


In [ ]:
# Price — strip $ and , first
df['_price_check'] = df['price'].str.replace(r'[\$,]', '', regex=True).str.strip()
df['_price_check'] = pd.to_numeric(df['_price_check'], errors='coerce')

df['_min_nights_check']  = pd.to_numeric(df['minimum nights'],   errors='coerce')
df['_avail_check']       = pd.to_numeric(df['availability 365'], errors='coerce')
df['_reviews_check']     = pd.to_numeric(df['number of reviews'], errors='coerce')

summary = pd.DataFrame({
    'Column'  : ['price', 'minimum nights', 'availability 365', 'number of reviews'],
    'Min'     : [df['_price_check'].min(), df['_min_nights_check'].min(),
                 df['_avail_check'].min(), df['_reviews_check'].min()],
    'Max'     : [df['_price_check'].max(), df['_min_nights_check'].max(),
                 df['_avail_check'].max(), df['_reviews_check'].max()],
    'Mean'    : [df['_price_check'].mean(), df['_min_nights_check'].mean(),
                 df['_avail_check'].mean(), df['_reviews_check'].mean()],
    'Nulls'   : [df['_price_check'].isnull().sum(), df['_min_nights_check'].isnull().sum(),
                 df['_avail_check'].isnull().sum(), df['_reviews_check'].isnull().sum()],
    'Problem?': ['No — $50 to $1,200 is OK',
                 'YES — negative values & max 5,645 impossible',
                 'YES — values above 365 impossible, negatives too',
                 'No — 0 to 1,024 is reasonable']
})
display(summary)

# Drop the temp check columns
df.drop(columns=['_price_check','_min_nights_check','_avail_check','_reviews_check'], inplace=True)

In [ ]:
# Outlier visualisation — minimum nights before cleaning
df['_min_n'] = pd.to_numeric(df['minimum nights'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['_min_n'].dropna(), bins=100, color='#378ADD', edgecolor='white')
axes[0].set_title('Minimum Nights — Raw (with outliers)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Minimum Nights')
axes[0].set_ylabel('Count')
axes[0].axvline(x=0,   color='#E24B4A', linestyle='--', label='0 (negative values exist below)')
axes[0].axvline(x=365, color='#EF9F27', linestyle='--', label='365 (values above this are outliers)')
axes[0].legend()

# Zoomed view 0-100 to see real distribution
clean_view = df['_min_n'][(df['_min_n'] >= 1) & (df['_min_n'] <= 100)]
axes[1].hist(clean_view, bins=50, color='#1D9E75', edgecolor='white')
axes[1].set_title('Minimum Nights — Realistic Range (1–100)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Minimum Nights')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Negative minimum nights : {(df['_min_n'] < 1).sum()} rows")
print(f"Minimum nights > 365    : {(df['_min_n'] > 365).sum()} rows")
df.drop(columns=['_min_n'], inplace=True)

---
## 8. Data Cleaning

Now we clean step by step. Each step is clearly documented  
so you can trace every decision.

---
### Step 8.1 — Drop Useless Columns


In [ ]:
print("Columns BEFORE drop:", df.shape[1])
print()

cols_to_drop = {
    'license'      : '99.99% null — only 2 non-null rows in 102,599',
    'house_rules'  : '50.8% null — free text, no analytical value',
    'country'      : 'Zero variance — every row is United States',
    'country code' : 'Zero variance — every row is US',
    'NAME'         : 'Listing title text — not needed for booking analysis'
}

for col, reason in cols_to_drop.items():
    print(f"  Dropping '{col}': {reason}")

df.drop(columns=list(cols_to_drop.keys()), inplace=True)

print()
print(f"Columns AFTER drop : {df.shape[1]}")
print(f"Remaining columns  : {df.columns.tolist()}")

### Step 8.2 — Remove Duplicate Rows

In [ ]:
before = len(df)

df.drop_duplicates(subset=['id'], keep='first', inplace=True)

removed = before - len(df)
print(f"Rows before : {before:,}")
print(f"Rows after  : {len(df):,}")
print(f"Removed     : {removed} duplicate rows (kept first occurrence)")

### Step 8.3 — Fix Neighbourhood Group Typos

In [ ]:
print("Before fix:")
print(df['neighbourhood group'].value_counts(dropna=False))
print()

typo_map = {
    'brookln'  : 'Brooklyn',
    'manhatan' : 'Manhattan'
}
df['neighbourhood group'] = df['neighbourhood group'].replace(typo_map)

print("After fix:")
print(df['neighbourhood group'].value_counts(dropna=False))

### Step 8.4 — Clean Price & Service Fee (Remove $ and ,)

In [ ]:
for col in ['price', 'service fee']:
    df[col] = df[col].str.replace(r'[\$,]', '', regex=True).str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Price — sample values after cleaning:")
print(df['price'].head(8).tolist())
print()
print(f"Price nulls     : {df['price'].isnull().sum()}")
print(f"Service fee nulls: {df['service fee'].isnull().sum()}")

### Step 8.5 — Convert All Numeric Columns

In [ ]:
numeric_cols = [
    'minimum nights',
    'number of reviews',
    'reviews per month',
    'review rate number',
    'calculated host listings count',
    'availability 365',
    'Construction year',
    'lat',
    'long'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Dtypes after conversion:")
print(df.dtypes)

### Step 8.6 — Handle Null Values

**Strategy:**
- **Drop rows** where critical columns are null — can't analyse without them
- **Fill** non-critical nulls with sensible defaults


In [ ]:
# ── Drop rows where critical columns are null ──
critical_cols = ['price', 'availability 365', 'neighbourhood group',
                 'neighbourhood', 'room type', 'lat', 'long']

before = len(df)
df.dropna(subset=critical_cols, inplace=True)
print(f"Dropped {before - len(df):,} rows with nulls in critical columns: {critical_cols}")

In [ ]:
# ── Fill non-critical nulls ──
fills = {
    'reviews per month'             : 0,
    'number of reviews'             : 0,
    'minimum nights'                : df['minimum nights'].median(),
    'review rate number'            : df['review rate number'].median(),
    'calculated host listings count': 1,
    'host_identity_verified'        : 'unconfirmed',
    'cancellation_policy'           : 'moderate',
    'instant_bookable'              : 'FALSE',
    'Construction year'             : df['Construction year'].median(),
    'service fee'                   : df['service fee'].median(),
    'host name'                     : 'Unknown',
    'last review'                   : 'No reviews',
}

for col, val in fills.items():
    n = df[col].isnull().sum()
    if n > 0:
        df[col].fillna(val, inplace=True)
        print(f"  Filled {n:>5} nulls in '{col}'  →  {repr(round(val, 2) if isinstance(val, float) else val)}")

print()
print("Remaining nulls after filling:")
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else "None — all nulls handled.")

### Step 8.7 — Remove Outliers

In [ ]:
before = len(df)

# minimum nights: must be 1–365 (negative = data error, >365 = unrealistic)
df = df[df['minimum nights'] >= 1]
df = df[df['minimum nights'] <= 365]

# availability 365: physically impossible to exceed 365
df = df[(df['availability 365'] >= 0) & (df['availability 365'] <= 365)]

after = len(df)
print(f"Rows before outlier removal : {before:,}")
print(f"Rows after outlier removal  : {after:,}")
print(f"Outlier rows removed        : {before - after:,}")
print()
print("minimum nights range now:", df['minimum nights'].min(), "to", df['minimum nights'].max())
print("availability 365 range now:", df['availability 365'].min(), "to", df['availability 365'].max())

### Step 8.8 — Rename Columns (PostgreSQL-friendly snake_case)

In [ ]:
df.rename(columns={
    'id'                            : 'listing_id',
    'host id'                       : 'host_id',
    'host name'                     : 'host_name',
    'neighbourhood group'           : 'borough',
    'neighbourhood'                 : 'neighbourhood',
    'lat'                           : 'latitude',
    'long'                          : 'longitude',
    'room type'                     : 'room_type',
    'Construction year'             : 'construction_year',
    'service fee'                   : 'service_fee',
    'minimum nights'                : 'minimum_nights',
    'number of reviews'             : 'number_of_reviews',
    'last review'                   : 'last_review',
    'reviews per month'             : 'reviews_per_month',
    'review rate number'            : 'review_rate',
    'calculated host listings count': 'host_listings_count',
    'availability 365'              : 'availability_365',
    'host_identity_verified'        : 'host_verified',
}, inplace=True)

print("Final column names:")
for col in df.columns:
    print(f"  {col}")

---
## 9. Cleaning Summary & Final Dataset Overview


In [ ]:
print("=" * 55)
print("  CLEANING SUMMARY")
print("=" * 55)
print(f"  Original rows   : 102,599")
print(f"  Final rows      : {len(df):,}")
print(f"  Rows removed    : {102599 - len(df):,}")
print(f"  Original cols   : 26")
print(f"  Final cols      : {df.shape[1]}")
print("=" * 55)
print()
print("Issues fixed:")
print("  [x] 541 duplicate rows removed")
print("  [x] 5 useless columns dropped")
print("  [x] 'brookln' and 'manhatan' typos corrected")
print("  [x] Price & service fee: '$' and ',' stripped, converted to float")
print("  [x] All numeric columns properly typed")
print("  [x] Nulls in critical columns → rows dropped")
print("  [x] Nulls in non-critical columns → filled with median/mode/default")
print("  [x] Negative minimum nights removed")
print("  [x] minimum nights > 365 removed")
print("  [x] availability_365 > 365 or < 0 removed")
print("  [x] All columns renamed to PostgreSQL snake_case")

In [ ]:
# Final dtypes
print("Final column types:")
print(df.dtypes)

In [ ]:
# Final null check
final_nulls = df.isnull().sum()
print("Final null check:")
if final_nulls.sum() == 0:
    print("  No nulls remaining in any column.")
else:
    print(final_nulls[final_nulls > 0])

In [ ]:
# Final data sample
df.head(5)

In [ ]:
# Final statistical summary
df.describe().round(2)

---
## 10. Post-Cleaning Visualisations

Quick visual check that the cleaned data looks correct before exporting.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cleaned Dataset — Key Distributions', fontsize=16, fontweight='bold', y=1.01)

# 1. Borough distribution
borough_counts = df['borough'].value_counts()
axes[0,0].bar(borough_counts.index, borough_counts.values,
              color=['#378ADD','#1D9E75','#EF9F27','#E24B4A','#7F77DD'])
axes[0,0].set_title('Listings by Borough', fontweight='bold')
axes[0,0].set_ylabel('Count')
for i, v in enumerate(borough_counts.values):
    axes[0,0].text(i, v + 150, f'{v:,}', ha='center', fontsize=9)

# 2. Room type distribution
room_counts = df['room_type'].value_counts()
colors_r = ['#378ADD','#1D9E75','#EF9F27','#E24B4A']
axes[0,1].pie(room_counts.values, labels=room_counts.index,
              autopct='%1.1f%%', colors=colors_r, startangle=90)
axes[0,1].set_title('Room Type Split', fontweight='bold')

# 3. Price distribution (cleaned)
axes[1,0].hist(df['price'].dropna(), bins=60, color='#1D9E75', edgecolor='white')
axes[1,0].set_title('Price Distribution (Cleaned)', fontweight='bold')
axes[1,0].set_xlabel('Price ($)')
axes[1,0].set_ylabel('Count')
axes[1,0].axvline(df['price'].mean(),   color='#E24B4A', linestyle='--',
                  label=f"Mean: ${df['price'].mean():.0f}")
axes[1,0].axvline(df['price'].median(), color='#EF9F27', linestyle='--',
                  label=f"Median: ${df['price'].median():.0f}")
axes[1,0].legend()

# 4. Availability 365 distribution (cleaned)
axes[1,1].hist(df['availability_365'].dropna(), bins=60, color='#378ADD', edgecolor='white')
axes[1,1].set_title('Availability 365 (Cleaned)', fontweight='bold')
axes[1,1].set_xlabel('Days Available per Year')
axes[1,1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Average price by borough — clean look
fig, ax = plt.subplots(figsize=(10, 5))

avg_price = df.groupby('borough')['price'].mean().sort_values(ascending=False)
bars = ax.bar(avg_price.index, avg_price.values,
              color=['#378ADD','#1D9E75','#EF9F27','#E24B4A','#7F77DD'])
ax.set_title('Average Listing Price by Borough (Cleaned Data)', fontsize=13, fontweight='bold')
ax.set_ylabel('Average Price ($)')
ax.set_xlabel('Borough')

for bar, val in zip(bars, avg_price.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'${val:.0f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Booking demand proxy: avg (365 - availability) = estimated booked days
df['est_booked_days'] = 365 - df['availability_365']

fig, ax = plt.subplots(figsize=(10, 5))
demand = df.groupby('borough')['est_booked_days'].mean().sort_values(ascending=False)
bars = ax.bar(demand.index, demand.values,
              color=['#1D9E75','#378ADD','#EF9F27','#E24B4A','#7F77DD'])
ax.set_title('Estimated Booked Days per Year by Borough\n(365 - availability_365)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Estimated Booked Days')
ax.set_xlabel('Borough')

for bar, val in zip(bars, demand.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f} days', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()
print()
print("This is your key booking demand insight — ready for deeper SQL analysis!")

---
## 11. Export Cleaned Dataset

Save the cleaned DataFrame as a CSV checkpoint.  
This is what we will load into **PostgreSQL** in the next step.


In [ ]:
df.drop(columns=['est_booked_days'], inplace=True)  # drop temp column before export

output_path = 'Airbnb_Cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"  Rows    : {len(df):,}")
print(f"  Columns : {df.shape[1]}")
print()
print("Final columns exported:")
for col in df.columns:
    print(f"  {col:<35} {str(df[col].dtype)}")

---
## 12. Load into PostgreSQL

> **Before running this cell:**
> 1. Make sure PostgreSQL is running locally
> 2. Create a database: `CREATE DATABASE airbnb_db;`
> 3. Update the credentials below

```bash
pip install sqlalchemy psycopg2-binary
```


In [ ]:
from sqlalchemy import create_engine

# ── UPDATE THESE WITH YOUR CREDENTIALS ──
DB_USER     = 'your_username'
DB_PASSWORD = 'your_password'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'airbnb_db'
TABLE_NAME  = 'listings'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

print(f"Loading {len(df):,} rows into PostgreSQL table '{TABLE_NAME}'...")

df.to_sql(
    TABLE_NAME,
    engine,
    if_exists='replace',   # replace if table already exists
    index=False,
    method='multi',        # batch insert — faster
    chunksize=1000
)

print(f"Done! Table '{TABLE_NAME}' is ready in database '{DB_NAME}'.")
print()
print("You can now open pgAdmin or psql and run:")
print(f"  SELECT COUNT(*) FROM {TABLE_NAME};")
print(f"  SELECT * FROM {TABLE_NAME} LIMIT 5;")

---
## 13. What's Next — SQL Analysis in PostgreSQL

Now that the data is clean and loaded, the next step is **SQL analysis**  
for the booking demand investigation.

### Key questions to answer with SQL:

```sql
-- 1. Which borough has the highest booking demand?
SELECT borough,
       ROUND(AVG(365 - availability_365), 1) AS avg_booked_days
FROM listings
GROUP BY borough
ORDER BY avg_booked_days DESC;

-- 2. Which room type is booked the most?
SELECT room_type,
       ROUND(AVG(reviews_per_month), 2)  AS avg_reviews_per_month,
       ROUND(AVG(365 - availability_365)) AS avg_booked_days
FROM listings
GROUP BY room_type
ORDER BY avg_booked_days DESC;

-- 3. Do instant-bookable listings get more demand?
SELECT instant_bookable,
       COUNT(*)                                    AS total_listings,
       ROUND(AVG(reviews_per_month), 2)            AS avg_reviews_per_month,
       ROUND(AVG(365 - availability_365))          AS avg_booked_days
FROM listings
GROUP BY instant_bookable;

-- 4. Top 10 neighbourhoods by booking demand
SELECT neighbourhood,
       borough,
       COUNT(*)                           AS listings,
       ROUND(AVG(365 - availability_365)) AS avg_booked_days,
       ROUND(AVG(price))                  AS avg_price
FROM listings
GROUP BY neighbourhood, borough
ORDER BY avg_booked_days DESC
LIMIT 10;

-- 5. Booking demand by cancellation policy
SELECT cancellation_policy,
       ROUND(AVG(reviews_per_month), 2)   AS avg_reviews,
       ROUND(AVG(365 - availability_365)) AS avg_booked_days
FROM listings
GROUP BY cancellation_policy
ORDER BY avg_booked_days DESC;
```

---

**Project flow complete:**
```
Raw CSV  →  Python EDA & Cleaning  →  Airbnb_Cleaned.csv  →  PostgreSQL  →  SQL Analysis  →  Dashboard
```
